# exp05d - Pengembangan AR-LRX: gerbang terpelajar vs gerbang tabel## Dari mana ide ini datangTabel segmen exp05b memberi petunjuk yang jelas. Pada V3, AR-LRX **kalah** dariXGBoost polos di Selasa (+12,5%), Kamis (+13,7%), hari promo (+1,8%) dan toko kecil(+5,6%), tetapi **menang besar** di Sabtu (-17,9%), Senin (-10,6%) dan Minggu (-7,2%).Satu nilai `w` global jelas kompromi buruk untuk pola sebeda itu.Ablasi bersih exp05b juga menunjukkan gerbang skalar praktis tidak menyumbang apa pun(rata-rata +0,075% untuk tahap pertama yang baik). Jadi persoalannya bukan "gerbangnyakurang disetel", melainkan **bentuk gerbangnya terlalu kaku**.Eksperimen ini menguji dua cara memperbaikinya, dan sengaja mengadu keduanya.| | Gagasan | Bentuk gerbang ||---|---|---|| **Seg** | `w` dipilih per segmen (promo / hari / hari x promo / kuartil toko) | tabel pencarian, konstan-sepotong || **Aug** | prediksi tahap pertama dijadikan **fitur** bagi tahap kedua | terpelajar, kontinu, bergantung fitur |Gagasan **Aug** secara teoretis lebih kuat: bila tahap kedua mengetahui `S1(x)`, iadapat mempelajari sendiri di mana tahap pertama lemah dan seberapa besar koreksi yangpantas - sebuah **gerbang terpelajar** yang secara ketat lebih umum daripada tabel`w(s)`. Desainnya faktorial 2x2 sehingga sumbangan masing-masing terpisah.## Penjinakan overfitting seleksiMemilih satu `w` per segmen memperbesar ruang seleksi dari 1 menjadi maksimal 14parameter. Uji coba awal menunjukkan gejalanya nyata: RMSE validation turun, RMSE testjustru naik sampai 3%. Ini persoalan yang sama yang sudah terlihat pada exp05b (V3dengan `S1 = linear` memburuk 7,5% semata-mata karena `w` ikut dipilih).Karena itu skema segmentasi dan hyperparameter **tidak** dipilih dari RMSE validationlangsung, melainkan dari **RMSE validasi-silang 5 lipatan kronologis di dalamvalidation**. Skema yang hanya mencocokkan derau tidak akan terpilih, dan skema`global` (setara gerbang skalar) menang secara wajar bila segmentasi tidak membantu.Peta `w` akhir baru dipasang pada seluruh validation setelah skemanya terpilih.Test tetap disentuh tepat satu kali.## EfisiensiPembanding (naif, XGBoost polos, kerangka lama, `S1` sendirian, AR-LRX skalar) **tidakdilatih ulang** - prediksinya dibaca dari `exp05b_..._predictions.npz`. Inilah gunanyamenyimpan prediksi pada exp05b. Yang dilatih hanya 18 model baru.## Prasyarat`src/experiments/arlrx.py` versi terbaru (memuat `run_arlrx_segmented`), lalu**Restart Kernel**. Penambahannya murni fungsi baru; `run_arlrx` tidak disentuhsehingga exp05a/exp05b tetap tereproduksi persis.

In [ ]:
import sys, os, json, warningssys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))warnings.filterwarnings("ignore")import numpy as npimport pandas as pdimport matplotlib.pyplot as pltfrom src.experiments import protocol as Pfrom src.experiments import arlrx as AP.set_global_seed()EXPERIMENT = "exp05d_rossmann_arlrx_dev"REF = "exp05b_rossmann_arlrx_audit"pd.set_option("display.width", 250); pd.set_option("display.max_columns", 60)print("Lingkungan:", P.environment_stamp())import inspectsrc = inspect.getsource(A)assert "run_arlrx_segmented" in src and "augment_stage1" in src, (    "arlrx.py masih versi lama. Ganti berkasnya, lalu Restart Kernel.")print("arlrx.py: versi dengan gerbang segmen + augmentasi  OK")print("Skema segmentasi:", A.SEGMENT_SCHEMES, "| lipatan seleksi:", A.N_GATE_FOLDS)

## 1. Data, dan pembanding yang dibaca dari exp05b

In [ ]:
frame = P.build_rossmann_frame("../data/raw/rossmann/train.csv",                               "../data/raw/rossmann/store.csv")VARIANTS = ["V1_customers_dropped", "V2_customers_lagged", "V3_sales_lagged"]QUICK_RUN = Falseif QUICK_RUN:    keep = np.sort(frame["Store"].unique())[:80]    frame = frame[frame["Store"].isin(keep)].reset_index(drop=True)    XGB_GRID = {"n_estimators": [300], "max_depth": [6], "learning_rate": [0.1],                "subsample": [0.8], "colsample_bytree": [0.8], "max_bin": [256]}else:    XGB_GRID = A.GRID_XGB_ARLRXdatasets = {v: P.build_rossmann_dataset(frame, v, target="log1p") for v in VARIANTS}print("Bentuk kerangka:", frame.shape, "| toko:", frame["Store"].nunique())ref = pd.read_csv(f"../results/{REF}.csv")npz = np.load(f"../results/{REF}_predictions.npz")print(f"Pembanding exp05b dimuat: {len(ref)} baris, {len(npz.files)} larik prediksi")# sanity: split identikfor v in VARIANTS:    n_ref = int(ref[ref.feature_set == v].n_test.iloc[0])    assert n_ref == len(datasets[v].y_test), f"n_test berbeda pada {v}"    assert np.allclose(npz[f"{v}|y_test"], datasets[v].y_test, atol=1e-5), \        f"y_test berbeda pada {v} -- exp05b dan exp05d tidak sebanding"print("Split dan target test identik dengan exp05b  OK")

## 2. Menjalankan 18 model baruTiga jenis pengembangan x dua tahap pertama x tiga varian. `S1 = linear` sengajatidak disertakan: exp05b sudah menunjukkan ia kalah dari XGBoost polos di ketigavarian, dan mempertahankannya hanya menambah waktu tanpa menambah informasi.* **AR-LRX-Seg** - gerbang per segmen, tanpa augmentasi* **AR-LRX-Aug** - augmentasi, gerbang skalar (`schemes=("global",)`)* **AR-LRX-Aug-Seg** - keduanya

In [ ]:
KINDS = ("structural", "struct_linear")rows = []for variant in VARIANTS:    d = datasets[variant]    for kind in KINDS:        rows.append(A.run_arlrx_segmented(f"AR-LRX-Seg [{kind}]", d, kind, XGB_GRID,                                          inverse_transform=np.expm1))        rows.append(A.run_arlrx_segmented(f"AR-LRX-Aug [{kind}]", d, kind, XGB_GRID,                                          schemes=("global",), augment_stage1=True,                                          inverse_transform=np.expm1))        rows.append(A.run_arlrx_segmented(f"AR-LRX-Aug-Seg [{kind}]", d, kind, XGB_GRID,                                          augment_stage1=True,                                          inverse_transform=np.expm1))        print(f"  {variant:24s} S1={kind:14s} selesai", flush=True)results = P.save_results(rows, EXPERIMENT)new_key = {(r["feature_set"], r["model"]): r for r in rows}print(f"\n{len(results)} baris ditulis ke ../results/{EXPERIMENT}.csv")

In [ ]:
# Simpan prediksi model baru juga.store = {}for (variant, model), r in new_key.items():    tag = f"{variant}|{model}"    for k in ("_test_pred", "_stage1_test_raw", "_test_correction", "_test_w"):        if k in r:            store[f"{tag}|{k[1:]}"] = np.asarray(r[k], dtype=np.float32)path = f"../results/{EXPERIMENT}_predictions.npz"np.savez_compressed(path, **store)print(f"{len(store)} larik disimpan ({os.path.getsize(path)/1e6:.1f} MB)")

## 3. Tabel utama - semua kandidat, satu peringkatBaris exp05b dimasukkan apa adanya; tidak ada yang dilatih ulang, sehinggaperbandingannya berada di protokol yang sama persis.

In [ ]:
def pred_of(variant, model):    k = f"{variant}|{model}|test_pred"    if k in npz.files:        return np.asarray(npz[k], dtype=float)    return np.asarray(new_key[(variant, model)]["_test_pred"], dtype=float)REF_MODELS = ["SeasonalNaive(store x dow x promo median)", "XGBoost",              "LR-XGB (residual, tanpa gerbang)"] + \             [f"S1 [{k}]" for k in KINDS] + [f"AR-LRX [{k}]" for k in KINDS]table = []for variant in VARIANTS:    sub_ref = ref[ref.feature_set == variant].set_index("model")    for mname in REF_MODELS:        table.append({"varian": variant, "model": mname, "asal": "exp05b",                      "RMSE": sub_ref.loc[mname, "orig_RMSE"],                      "MAE": sub_ref.loc[mname, "orig_MAE"],                      "RMSPE": sub_ref.loc[mname, "orig_RMSPE"],                      "R2": sub_ref.loc[mname, "orig_R2"]})    for kind in KINDS:        for pre in ("AR-LRX-Seg", "AR-LRX-Aug", "AR-LRX-Aug-Seg"):            r = new_key[(variant, f"{pre} [{kind}]")]            table.append({"varian": variant, "model": f"{pre} [{kind}]", "asal": "exp05d",                          "RMSE": r["orig_RMSE"], "MAE": r["orig_MAE"],                          "RMSPE": r["orig_RMSPE"], "R2": r["orig_R2"]})table = pd.DataFrame(table)table.to_csv(f"../results/{EXPERIMENT}_main.csv", index=False)for variant in VARIANTS:    t = table[table.varian == variant].sort_values("RMSE").reset_index(drop=True)    t.index = t.index + 1    print(f"\n--- {variant} ---")    display(t[["model", "asal", "RMSE", "MAE", "RMSPE", "R2"]].round(5).to_string())

## 4. Ablasi faktorial 2x2 - augmentasi vs segmentasiEmpat sel per (varian, tahap pertama). Sel `skalar` diambil dari exp05b.

In [ ]:
cells = []for variant in VARIANTS:    sub_ref = ref[ref.feature_set == variant].set_index("model")    for kind in KINDS:        base = sub_ref.loc[f"AR-LRX [{kind}]", "orig_RMSE"]        seg  = new_key[(variant, f"AR-LRX-Seg [{kind}]")]["orig_RMSE"]        aug  = new_key[(variant, f"AR-LRX-Aug [{kind}]")]["orig_RMSE"]        both = new_key[(variant, f"AR-LRX-Aug-Seg [{kind}]")]["orig_RMSE"]        cells.append({"varian": variant, "S1": kind,                      "skalar (exp05b)": base, "+Seg": seg, "+Aug": aug, "+Aug+Seg": both,                      "efek Seg (%)": (seg - base) / base * 100,                      "efek Aug (%)": (aug - base) / base * 100,                      "efek keduanya (%)": (both - base) / base * 100})abl = pd.DataFrame(cells)abl.to_csv(f"../results/{EXPERIMENT}_ablation_2x2.csv", index=False)print("Negatif = lebih baik daripada gerbang skalar exp05b.")display(abl.round(3).to_string(index=False))print("\nRata-rata efek tiap komponen:")display(abl[["efek Seg (%)", "efek Aug (%)", "efek keduanya (%)"]].mean().round(3))print("\nSkema segmentasi yang terpilih (perhatikan berapa kali jatuh ke 'global'):")sc = pd.DataFrame([{"varian": v, "model": m,                    "skema": r["segment_scheme"], "n_segmen": r["n_segments"],                    "w": r["gate_w_map"]}                   for (v, m), r in new_key.items() if "Seg" in m])display(sc.sort_values(["varian", "model"]).to_string(index=False))

## 5. Uji Diebold-Mariano dua skala untuk model terbaikPembandingnya lima: XGBoost polos, naif per toko, kerangka lama, tahap pertamasendirian, dan **AR-LRX skalar exp05b** - yang terakhir mengukur apakah pengembanganini benar-benar melampaui versi sebelumnya, bukan sekadar setara.

In [ ]:
best_per_variant = {}for variant in VARIANTS:    t = table[(table.varian == variant) & (table.asal == "exp05d")]    best_per_variant[variant] = t.loc[t.RMSE.idxmin(), "model"]print("Model exp05d terbaik per varian:")for v, m in best_per_variant.items():    print(f"  {v:24s} -> {m}")dm_rows = []for variant in VARIANTS:    d = datasets[variant]    y_log = d.y_test; y_org = np.expm1(y_log)    bm = best_per_variant[variant]    kind = bm.split("[")[1].rstrip("]")    p_new = pred_of(variant, bm)    refs = [("XGBoost polos", "XGBoost"),            ("Naif per toko", "SeasonalNaive(store x dow x promo median)"),            ("Kerangka lama", "LR-XGB (residual, tanpa gerbang)"),            (f"S1 [{kind}] sendirian", f"S1 [{kind}]"),            (f"AR-LRX skalar [{kind}] (exp05b)", f"AR-LRX [{kind}]")]    for label, mname in refs:        p_ref = pred_of(variant, mname)        tl = P.diebold_mariano(y_log, p_new, p_ref)        to = P.diebold_mariano(y_org, np.expm1(p_new), np.expm1(p_ref))        dm_rows.append({"varian": variant, "model": bm, "pembanding": label,                        "DM (log)": round(tl["DM"], 3), "p (log)": tl["p_value"],                        "DM (asli)": round(to["DM"], 3), "p (asli)": to["p_value"],                        "menang log": bool(tl["DM"] < 0 and tl["p_value"] < 0.05),                        "menang asli": bool(to["DM"] < 0 and to["p_value"] < 0.05),                        "sepakat": bool((tl["DM"] < 0) == (to["DM"] < 0))})dm = pd.DataFrame(dm_rows)dm.to_csv(f"../results/{EXPERIMENT}_dm.csv", index=False)display(dm.to_string(index=False))print(f"\nSepakat arah      : {int(dm.sepakat.sum())}/{len(dm)}")print(f"Menang sig (asli) : {int(dm['menang asli'].sum())}/{len(dm)}")lost = dm[~dm["menang asli"]]if len(lost):    print("\nBELUM menang signifikan pada skala asli:")    display(lost[["varian", "pembanding", "DM (asli)", "p (asli)"]].to_string(index=False))

## 6. Apakah kelemahan segmen exp05b tertutup?exp05b menunjukkan AR-LRX kalah dari XGBoost polos pada segmen tertentu (V3: Selasa,Kamis, promo, toko kecil). Sel ini memeriksa apakah pengembangan menutupnya.

In [ ]:
seg_rows = []for variant in VARIANTS:    d = datasets[variant]    y = np.expm1(d.y_test)    x = np.expm1(pred_of(variant, "XGBoost"))    kind = best_per_variant[variant].split("[")[1].rstrip("]")    old = np.expm1(pred_of(variant, f"AR-LRX [{kind}]"))    new = np.expm1(pred_of(variant, best_per_variant[variant]))    fn = list(d.feature_names); Xt = d.X_test    promo = Xt[:, fn.index("Promo")] if "Promo" in fn else np.zeros(len(y))    dow = Xt[:, fn.index("DayOfWeek")] if "DayOfWeek" in fn else np.zeros(len(y))    def rm(t, p, m):        return float(np.sqrt(np.mean((t[m] - p[m]) ** 2))) if m.sum() else np.nan    segs = {"semua": np.ones(len(y), bool), "promo": promo == 1, "tanpa promo": promo == 0}    for dv in sorted(set(np.unique(dow).tolist()))[:7]:        segs[f"hari {int(dv)}"] = dow == dv    if "Store" in fn:        st = Xt[:, fn.index("Store")]        mean_by_store = pd.Series(y).groupby(pd.Series(st)).transform("mean").to_numpy()        q = pd.Categorical(pd.qcut(mean_by_store, 4,                                   labels=["Q1 kecil", "Q2", "Q3", "Q4 besar"],                                   duplicates="drop"))        for lab in q.categories:            segs[f"toko {lab}"] = np.asarray(q == lab)    for name, m in segs.items():        ro, rn, rx = rm(y, old, m), rm(y, new, m), rm(y, x, m)        seg_rows.append({"varian": variant, "segmen": name, "n": int(m.sum()),                         "AR-LRX lama": ro, "exp05d": rn, "XGBoost": rx,                         "lama vs XGB (%)": (ro - rx) / rx * 100,                         "baru vs XGB (%)": (rn - rx) / rx * 100,                         "perbaikan (pp)": ((rn - rx) / rx * 100) - ((ro - rx) / rx * 100)})seg = pd.DataFrame(seg_rows)seg.to_csv(f"../results/{EXPERIMENT}_segments.csv", index=False)print("Negatif = lebih baik dari XGBoost. 'perbaikan (pp)' negatif = exp05d menutup celah.")for v in VARIANTS:    print(f"\n--- {v} ---")    display(seg[seg.varian == v].round(3).to_string(index=False))fixed = seg[(seg["lama vs XGB (%)"] > 0) & (seg["baru vs XGB (%)"] < 0)]print(f"\nSegmen yang berbalik dari kalah menjadi menang: {len(fixed)}")if len(fixed):    display(fixed[["varian", "segmen", "n", "lama vs XGB (%)", "baru vs XGB (%)"]]            .round(3).to_string(index=False))

## 7. Determinisme dan ringkasan naskah

In [ ]:
P.set_global_seed()v0 = VARIANTS[0]again = A.run_arlrx_segmented("AR-LRX-Aug [struct_linear]", datasets[v0], "struct_linear",                              XGB_GRID, schemes=("global",), augment_stage1=True,                              inverse_transform=np.expm1)first = new_key[(v0, "AR-LRX-Aug [struct_linear]")]print("Prediksi identik bit-per-bit:",      np.array_equal(again["_test_pred"], first["_test_pred"]))print(f"RMSE: {again['orig_RMSE']:.8f} vs {first['orig_RMSE']:.8f}")print("\n" + "=" * 88)for variant in VARIANTS:    bm = best_per_variant[variant]    r = new_key[(variant, bm)]    sub_ref = ref[ref.feature_set == variant].set_index("model")    kind = bm.split("[")[1].rstrip("]")    dsub = dm[dm.varian == variant].set_index("pembanding")    print(f"\n{variant}  ->  {bm}")    print(f"  RMSE {r['orig_RMSE']:.2f} | MAE {r['orig_MAE']:.2f} | "          f"RMSPE {r['orig_RMSPE']:.5f} | R2 {r['orig_R2']:.4f}")    for lab, mname in [("naif per toko", "SeasonalNaive(store x dow x promo median)"),                       ("XGBoost polos", "XGBoost"),                       ("kerangka lama", "LR-XGB (residual, tanpa gerbang)"),                       (f"AR-LRX skalar exp05b", f"AR-LRX [{kind}]")]:        base = sub_ref.loc[mname, "orig_RMSE"]        d_ = (r["orig_RMSE"] - base) / base * 100        key = {"naif per toko": "Naif per toko", "XGBoost polos": "XGBoost polos",               "kerangka lama": "Kerangka lama",               "AR-LRX skalar exp05b": f"AR-LRX skalar [{kind}] (exp05b)"}[lab]        p_ = dsub.loc[key, "p (asli)"]        arah = "lebih baik" if d_ < 0 else "LEBIH BURUK"        print(f"    {arah} {abs(d_):5.2f}% dari {lab:26s} (DM asli, p = {p_:.3g})")print("\n" + "=" * 88)print("Berkas yang dihasilkan:")for f in sorted(os.listdir("../results")):    if f.startswith(EXPERIMENT):        print(f"  ../results/{f}")

## 8. Cara melaporkan* **Desainnya faktorial, jadi laporkan keempat selnya** - termasuk bila `+Seg`  memburuk. Justru kontras "gerbang tabel gagal, gerbang terpelajar berhasil" yang  membuat temuannya bermakna: yang menentukan bukan banyaknya parameter gerbang,  melainkan apakah gerbang itu dapat bergantung pada fitur secara kontinu.* **Sebutkan penjinakan overfitting seleksi.** Bahwa skema segmentasi dipilih lewat  validasi-silang di dalam validation, dan bahwa tanpa itu RMSE test naik sampai 3%,  adalah bukti kedisiplinan metodologis yang jarang ditunjukkan naskah sejenis.* **Sebutkan keterbatasan augmentasi.** `S1` pada blok fit adalah nilai in-sample;  untuk penaksir struktural nilai itu sedikit optimistis karena rata-rata kelompok  memuat titiknya sendiri. Konvensi ini sama dengan yang dipakai membentuk residual  pada kerangka aslinya, tetapi tetap harus dinyatakan.* **Jangan buang exp05b.** Ia yang membuktikan reproduksi persis, kesepakatan dua  skala, dan ablasi gerbang skalar. exp05d berdiri di atasnya, bukan menggantikannya.